In [1]:
pip install google-adk

In [13]:
import os
import ast
import requests
from google.adk import Agent

PROJECT_ID = "qwiklabs-gcp-01-4295d9eedc89"
LOCATION = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION


In [14]:


def get_weather(city: str) -> dict:
    """Get current weather for a given city using the Open-Meteo API.

    Args:
        city: Name of the city, such as London or Washington DC.

    Returns:
        A dictionary containing current weather information
        or an error message.
    """
    print(f"TOOL CALLED: get_weather({city})")

    try:
        # Step 1: Convert city name into coordinates
        geo_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10
        )
        geo_response.raise_for_status()

        locations = geo_response.json().get("results", [])

        if not locations:
            return {"status": "error", "message": f"City not found: {city}"}

        location = locations[0]
        latitude = location["latitude"]
        longitude = location["longitude"]

        # Step 2: Request current weather
        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": (
                    "temperature_2m,relative_humidity_2m,"
                    "apparent_temperature,precipitation,"
                    "weather_code,wind_speed_10m"
                ),
                "temperature_unit": "fahrenheit",
                "wind_speed_unit": "mph",
                "timezone": "auto"
            },
            timeout=10
        )
        weather_response.raise_for_status()

        data = weather_response.json()

        return {
            "status": "success",
            "city": location["name"],
            "region": location.get("admin1"),
            "country": location.get("country"),
            "observation_time": data["current"]["time"],
            "timezone": data["timezone"],
            "temperature_f": data["current"]["temperature_2m"],
            "feels_like_f": data["current"]["apparent_temperature"],
            "humidity_percent": data["current"]["relative_humidity_2m"],
            "precipitation_mm": data["current"]["precipitation"],
            "wind_speed_mph": data["current"]["wind_speed_10m"],
            "weather_code": data["current"]["weather_code"]
        }

    except (requests.RequestException, ValueError, KeyError) as e:
        return {"status": "error", "message": str(e)}


In [15]:
get_weather("london")

TOOL CALLED: get_weather(london)


{'status': 'success',
 'city': 'London',
 'region': 'England',
 'country': 'United Kingdom',
 'observation_time': '2026-10-13T16:30',
 'timezone': 'Europe/London',
 'temperature_f': 67.5,
 'feels_like_f': 60.2,
 'humidity_percent': 43,
 'precipitation_mm': 0.0,
 'wind_speed_mph': 13.6,
 'weather_code': 0}

In [16]:


agent = Agent(
    name="WeatherBot",
    model="gemini-3.8-flash",
    instruction="""  When a user asks about current weather:
    1. Identify the city they are asking about.
    2. Call the get_weather tool.
    3. Use the tool's results to answer their question.
    4. Provide temperature, conditions, humidity,
       and wind speed when relevant.

    Never invent current weather information.
    If the tool returns an error, explain the problem.
    The weather_code is a WMO numeric code; do not
    guess its description if you are uncertain.""",
    tools=[get_weather]
)
print("Agent created:", agent.name)


Agent created: WeatherBot


In [17]:

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "weatherbot_app"
USER_ID = "workshop_user"

session_service = InMemorySessionService()

runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service
)

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID
)

SESSION_ID = session.id
print(f"Session created: {SESSION_ID}")


Session created: 24cc2a88-329d-4a8a-940a-d02c161434a8


In [18]:

async def ask_weatherbot(question):
    print(f"USER: {question}\n")

    message = types.Content(
        role="user",
        parts=[types.Part(text=question)]
    )

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=message
    ):
        if event.get_function_calls():
            for call in event.get_function_calls():
                print(f"FUNCTION CALL: {call.name}")
                print(f"ARGUMENTS: {call.args}\n")

        if event.get_function_responses():
            for response in event.get_function_responses():
                print(f"TOOL RESPONSE: {response.response}\n")

        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"WEATHERBOT: {part.text}")

await ask_weatherbot("What's the current weather in London?")


USER: What's the current weather in London?

FUNCTION CALL: get_weather
ARGUMENTS: {'city': 'London'}

TOOL CALLED: get_weather(London)
TOOL RESPONSE: {'status': 'success', 'city': 'London', 'region': 'England', 'country': 'United Kingdom', 'observation_time': '2026-10-13T16:30', 'timezone': 'Europe/London', 'temperature_f': 67.5, 'feels_like_f': 60.2, 'humidity_percent': 43, 'precipitation_mm': 0.0, 'wind_speed_mph': 13.6, 'weather_code': 0}

WEATHERBOT: The current weather in London, United Kingdom:

- **Temperature:** 67.5°F (Feels like 60.2°F)
- **Conditions:** Clear sky (WMO code 0)
- **Humidity:** 43%
- **Wind Speed:** 13.6 mph
- **Precipitation:** 0 mm


In [21]:

cities = ['New York', 'Paris', 'Tokyo', 'Cairo', 'Sydney']

In [22]:
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID
)
SESSION_ID = session.id

In [23]:

import asyncio

for city in cities:
    # Create a fresh session for this city
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID
    )

    SESSION_ID = session.id

    print(f"\n{'=' * 50}")
    print(f"City: {city}")
    print(f"New Session ID: {SESSION_ID}")

    # Ask the agent about this city
    try:
        await ask_weatherbot(
            f"What's the current weather in {city}?"
        )
    except Exception as e:
        print(f"Error processing {city}: {e}")

    # Space out requests to reduce 429 errors
    await asyncio.sleep(5)



City: New York
New Session ID: d32d8fdd-8ef6-4630-8e14-cf47210174e7
USER: What's the current weather in New York?

FUNCTION CALL: get_weather
ARGUMENTS: {'city': 'New York'}

TOOL CALLED: get_weather(New York)
TOOL RESPONSE: {'status': 'success', 'city': 'New York', 'region': 'New York', 'country': 'United States', 'observation_time': '2026-10-13T11:30', 'timezone': 'America/New_York', 'temperature_f': 66.8, 'feels_like_f': 68.2, 'humidity_percent': 84, 'precipitation_mm': 0.0, 'wind_speed_mph': 6.5, 'weather_code': 2}

WEATHERBOT: The current weather in New York is as follows:

* **Temperature:** 66.8°F (feels like 68.2°F)
* **Conditions:** Partly cloudy (weather code 2)
* **Humidity:** 84%
* **Wind Speed:** 6.5 mph
* **Precipitation:** 0 mm

City: Paris
New Session ID: d8b434b9-674e-4d6b-ae59-48fc40bc5716
USER: What's the current weather in Paris?

FUNCTION CALL: get_weather
ARGUMENTS: {'city': 'Paris'}

TOOL CALLED: get_weather(Paris)
TOOL RESPONSE: {'status': 'success', 'city': 'P